# 72 · Authz & governance — Apache Ranger column masking on Trino

**Ranger is where fine-grained access control lives for the mesh's query engine.** Trino can
federate every store and join across them (notebook `20`), and the marts are tested and documented
(notebook `40`) — but *who is allowed to see which column, in which row* is a separate question, and
the honest place to answer it is **at the query engine, not in each application**. If authz lived in
the notebook, the dashboard, and the BI tool separately, they would drift; put it in Trino and every
consumer inherits the same rules no matter how it connects.

That is what **Apache Ranger** does here. Trino 468 runs a **native Ranger access-control plugin** that
pulls policies from Ranger Admin every ~30s and enforces them on *every* query. Its model has three
moving parts:

> **Default-deny** — once the plugin loads, anything not explicitly granted is denied. A permissive
> base policy grants the `public` group access so mesh consumers aren't locked out.
>
> **Column masks** — a policy can rewrite a column's value *per user* (e.g. return `NULL` instead of
> the real number) without changing the query.
>
> **Row filters** — a policy can inject a `WHERE` predicate *per user*, so different users see
> different rows of the same table.

This notebook demonstrates the middle one against a **real, committed policy**: run the *same* `SELECT`
as two different Trino users and watch one column come back masked for one of them and unmasked for the
other. The policy is `mask-depression-pct-analyst` (codified in `scripts/ranger_setup.py`): user
`analyst` sees `depression_pct` as `NULL`; everyone else sees the real value.

> **Which Trino?** This runs against the **main** coordinator
> (`trino.data-mesh.svc.cluster.local:8080`) — the one with the Ranger plugin. There is *also* a
> `trino-noauth` proxy in the mesh that **bypasses Ranger entirely**; Soda (data quality) and some BI
> paths use it precisely because they are trusted pipelines that must see everything. We demo on the
> **main** Trino, where governance is live. More on that contrast at the end.

> **Read-only.** Every cell is a `SELECT`. Ranger is a policy *source* we read the effect of — the
> policies themselves are managed by `scripts/ranger_setup.py`, not by this notebook — so there is
> nothing to create and no cleanup section.

## Setup

The `trino` Python client is **not** in the singleuser base image (which ships `polars`, `s3fs`,
`pyarrow`, `duckdb`, `fastavro`), so we install it here — exactly as notebooks `20`/`40` do. `polars`,
used to render every result frame, already ships in the image.

In [1]:
%pip install -q trino


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Connect — one helper, called as different users

The whole demonstration turns on **one idea**: the Trino client chooses the *session user*, and Ranger
applies *that user's* policies. So instead of one connection, we build a small `q(user, sql)` helper
that opens a connection **as** a named user and runs a query. The same SQL, handed to `q('dbt', ...)`
vs `q('analyst', ...)`, is governed differently — that is the entire point of engine-side authz.

The main coordinator speaks **plain HTTP on 8080** and takes an **empty password** — there is no
secret here. Authz is *not* password-based: it is evaluated on the session user against Ranger's
policies. (`analyst` and `dbt` are both covered by the permissive `public` base policy, so both are
allowed to `SELECT` at all; the difference is only the column mask.)

Connection is env-driven with an **in-cluster default host**. Note we do **not** read a bare
`TRINO_PORT` env var: this notebook is validated from inside the `data-mesh` namespace, where
Kubernetes injects a service-discovery variable `TRINO_PORT=tcp://<clusterIP>:8080` for the `trino`
Service — `int()` of that string would fail. The port is fixed at 8080 for this coordinator, so we pin
it and leave only the host overridable.

`q` **catches a Ranger denial and returns it as a frame** rather than raising: under default-deny, a
denial is a governance *outcome* worth showing, not an error to leak.

In [2]:
import os
import trino
import polars as pl

# in-cluster default host; port pinned to 8080 (see note above re: the k8s TRINO_PORT collision)
TRINO_HOST = os.environ.get("TRINO_HOST", "trino.data-mesh.svc.cluster.local")
TRINO_PORT = 8080

# the table carrying the masked column, and the two users we contrast
MASKED_TABLE   = "iceberg.dbt.mart_state_health_trends"
MASKED_COLUMN  = "depression_pct"
USER_UNMASKED  = "dbt"       # not named in the mask policy -> sees the real value
USER_MASKED    = "analyst"   # named in mask-depression-pct-analyst -> sees NULL

def _conn(user):
    """Open a Trino connection AS `user`. Ranger enforces THAT user's policies -- the client
    picks the identity, the engine applies authz. Empty password: the main coordinator trusts
    the session user (LAN-only), and access is decided by Ranger, not a credential."""
    return trino.dbapi.connect(
        host=TRINO_HOST,
        port=TRINO_PORT,
        user=user,
        catalog="iceberg",
        schema="dbt",
        http_scheme="http",   # main coordinator is plain HTTP on 8080, Ranger plugin active
    )

def q(user, sql):
    """Run a SELECT as `user`, return a polars DataFrame. A Ranger denial (default-deny) is
    caught and returned as a one-row frame -- a governance outcome to display, not to raise."""
    try:
        cur = _conn(user).cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]
        return pl.DataFrame(rows, schema=cols, orient="row")
    except Exception as e:
        return pl.DataFrame({"user": [user], "denied_or_error": [str(e).splitlines()[0]]})

# prove the connection by what it queries back (as dbt), not by echoing the address
ver = q(USER_UNMASKED, "SELECT version() AS v").item(0, "v")
print(f"Trino        : {TRINO_HOST}:{TRINO_PORT}  (main coordinator, Ranger plugin active)")
print(f"Trino version: {ver}")
print(f"masked table : {MASKED_TABLE}   masked column: {MASKED_COLUMN}")
print(f"users        : unmasked='{USER_UNMASKED}'  masked='{USER_MASKED}'")

Trino        : trino.data-mesh.svc.cluster.local:8080  (main coordinator, Ranger plugin active)
Trino version: 468
masked table : iceberg.dbt.mart_state_health_trends   masked column: depression_pct
users        : unmasked='dbt'  masked='analyst'


## Baseline — the unmasked truth, as `dbt`

First, the ground truth. We query `mart_state_health_trends` (the BRFSS chronic-condition mart from
notebook `40`, one row per US state per year) as user **`dbt`**. `dbt` is **not** named in any mask
policy, so Ranger returns `depression_pct` exactly as stored. We select a couple of neighbouring
columns (`diabetes_pct`, `asthma_pct`) too, so that when we mask, it's obvious the mask is
**column-scoped** — those siblings must stay identical across both users.

In [3]:
SQL = f"""
    SELECT state, year, {MASKED_COLUMN}, diabetes_pct, asthma_pct
    FROM {MASKED_TABLE}
    WHERE year = 2024
    ORDER BY state
    LIMIT 8
"""

baseline = q(USER_UNMASKED, SQL)
print(f"as user '{USER_UNMASKED}' (not in any mask policy) -> real {MASKED_COLUMN} values:")
baseline

as user 'dbt' (not in any mask policy) -> real depression_pct values:


state,year,depression_pct,diabetes_pct,asthma_pct
str,i64,f64,f64,f64
"""AK""",2024,21.9,9.4,13.6
"""AL""",2024,24.8,15.1,13.85
"""AR""",2024,25.2,15.3,13.55
"""AZ""",2024,19.2,11.5,13.95
"""CA""",2024,17.8,12.6,11.15
"""CO""",2024,23.0,8.4,14.3
"""CT""",2024,18.4,11.7,14.15
"""DC""",2024,22.4,8.4,14.75


## Masked — the SAME query, as `analyst`

Now the identical SQL — same table, same columns, same filter — but run as user **`analyst`**. This
user *is* the subject of `mask-depression-pct-analyst`, a `MASK_NULL` data-mask policy on the
`depression_pct` column. Ranger rewrites the query on the way in so that column comes back `NULL`,
while every other column is untouched. Nothing about the query changed — only *who asked*.

In [4]:
masked = q(USER_MASKED, SQL)
print(f"as user '{USER_MASKED}' (mask-depression-pct-analyst applies) -> {MASKED_COLUMN} is NULL:")
masked

as user 'analyst' (mask-depression-pct-analyst applies) -> depression_pct is NULL:


state,year,depression_pct,diabetes_pct,asthma_pct
str,i64,null,f64,f64
"""AK""",2024,null,9.4,13.6
"""AL""",2024,null,15.1,13.85
"""AR""",2024,null,15.3,13.55
"""AZ""",2024,null,11.5,13.95
"""CA""",2024,null,12.6,11.15
"""CO""",2024,null,8.4,14.3
"""CT""",2024,null,11.7,14.15
"""DC""",2024,null,8.4,14.75


### Side by side — the mask in one frame

Join the two results on their grain `(state, year)` and put the two `depression_pct` columns next to
each other. `depression_pct_dbt` carries the real percentages; `depression_pct_analyst` is `NULL` for
every row; `analyst_masked` confirms it. The sibling columns (`diabetes_pct`, `asthma_pct`) are shown
to be **identical** across both users — proof the mask is scoped to exactly one column, not the row.

In [5]:
side = (
    baseline.select(
        "state", "year",
        pl.col(MASKED_COLUMN).alias(f"{MASKED_COLUMN}_dbt"),
        pl.col("diabetes_pct").alias("diabetes_pct_dbt"),
    )
    .join(
        masked.select(
            "state", "year",
            pl.col(MASKED_COLUMN).alias(f"{MASKED_COLUMN}_analyst"),
            pl.col("diabetes_pct").alias("diabetes_pct_analyst"),
        ),
        on=["state", "year"], how="inner",
    )
    .with_columns(
        pl.col(f"{MASKED_COLUMN}_analyst").is_null().alias("analyst_masked"),
        (pl.col("diabetes_pct_dbt") == pl.col("diabetes_pct_analyst")).alias("sibling_unchanged"),
    )
    .sort("state")
)

n_masked   = int(side.select(pl.col("analyst_masked").sum()).item())
n_siblings = int(side.select(pl.col("sibling_unchanged").sum()).item())
print(f"{MASKED_COLUMN} masked for analyst on {n_masked}/{side.height} rows; "
      f"sibling diabetes_pct identical on {n_siblings}/{side.height} rows")
side

depression_pct masked for analyst on 8/8 rows; sibling diabetes_pct identical on 8/8 rows


state,year,depression_pct_dbt,diabetes_pct_dbt,depression_pct_analyst,diabetes_pct_analyst,analyst_masked,sibling_unchanged
str,i64,f64,f64,null,f64,bool,bool
"""AK""",2024,21.9,9.4,null,9.4,true,true
"""AL""",2024,24.8,15.1,null,15.1,true,true
"""AR""",2024,25.2,15.3,null,15.3,true,true
"""AZ""",2024,19.2,11.5,null,11.5,true,true
"""CA""",2024,17.8,12.6,null,12.6,true,true
"""CO""",2024,23.0,8.4,null,8.4,true,true
"""CT""",2024,18.4,11.7,null,11.7,true,true
"""DC""",2024,22.4,8.4,null,8.4,true,true


## What the policy actually says

The demonstration above is the *effect*. Here is the *cause* — the exact policy, codified in
`scripts/ranger_setup.py` so it survives a Ranger DB rebuild (policies are otherwise UI/API state that
dies with the DB):

```python
# scripts/ranger_setup.py -- ensure_mask_policy()
pol = {"service": "trino", "name": "mask-depression-pct-analyst", "policyType": 1, "isEnabled": True,
       "resources": {"catalog": {"values": ["iceberg"]}, "schema": {"values": ["dbt"]},
                     "table": {"values": ["mart_state_health_trends"]},
                     "column": {"values": ["depression_pct"]}},
       "dataMaskPolicyItems": [{"users": ["analyst"], "accesses": [{"type": "select", "isAllowed": True}],
                                "dataMaskInfo": {"dataMaskType": "MASK_NULL"}}]}
```

- **`policyType: 1`** is Ranger's *data-mask* policy type (type `0` is plain access, type `2` is a row
  filter). The resource pins it to exactly one column: `iceberg.dbt.mart_state_health_trends.depression_pct`.
- **`dataMaskType: MASK_NULL`** replaces the value with `NULL` — which is exactly what `analyst` saw
  above. Other mask types Ranger supports include `MASK_HASH`, partial masks, and custom expressions;
  `MASK_NULL` is the bluntest and clearest.
- **`users: ["analyst"]`** scopes the mask to that one session user. Any *other* user (like `dbt`) is
  outside this policy and reads the real value.

### Default-deny + the permissive base policy

None of this would matter if consumers couldn't `SELECT` at all — and by default they can't. The moment
Trino loads the Ranger plugin, **unlisted access is denied**. The mesh's consumers connect as varied
users (`dbt`, `lightdash`, `trino` for DataHub, ...), and Ranger's stock service ships 13 default
policies that grant only to the single user `trino`. So `ranger_setup.py` **broadens every default
access policy to the `public` group** before the plugin is enabled — the anti-lockout step:

```python
# scripts/ranger_setup.py -- broaden_defaults_to_public()
# Add group `public` to every default ACCESS policy (policyType 0) -- anti-lockout for default-deny.
```

That is why both `analyst` and `dbt` could run the query at all: the permissive base grants the
`public` group `SELECT`, and *on top of that* the column-mask policy singles out `analyst`. Grant
broadly at the base, then restrict precisely with masks and filters — that layering is Ranger's model.

> If a policy change ever *did* deny one of these users, the `q()` helper would surface it as a
> `denied_or_error` frame rather than an exception — a denial is a governance result, and the notebook
> is built to *show* one, not choke on it.

## Row filters — not configured here (yet)

Ranger's third lever is the **row filter** (`policyType: 2`): a per-user `WHERE` predicate that shows
different users different *rows* of the same table, the row-wise sibling of the column mask above. It
is a natural next policy — e.g. restrict `analyst` to a subset of states — but the committed setup
(`scripts/ranger_setup.py`) currently defines **only** the column-mask policy plus the anti-lockout
base grants. There is no row-filter policy to demonstrate, so this section is intentionally a no-op
rather than a fabricated example. When one is added to `ensure_*` in that script, it would appear here
the same way the mask did: same query, two users, different rows returned.

## When to reach for Ranger

Authz at the query engine is the right tool when **many users share one data surface but must not all
see the same thing** — and the wrong tool when a single trusted pipeline just needs the data.

| reach for | when… | why |
|-----------|-------|-----|
| **Ranger-governed Trino** (`trino.data-mesh:8080`, this notebook) | multi-user access where columns/rows must be restricted per identity — analysts, BI users, ad-hoc consumers | one policy source, enforced on *every* query no matter the client; masks and row filters applied per session user; default-deny means access is explicit |
| **`trino-noauth` proxy** | a *trusted pipeline* that must see everything — Soda data-quality scans, some BI back-ends | it **bypasses Ranger entirely**; correct precisely because a governance layer would only get in the way of a system that is already trusted and needs the raw values |
| **store-native authz** (Postgres roles, etc.) | you're on a single store's own hot path, not federating | the engine-side policy layer doesn't apply when you're not going through Trino; use the store's own grants |

> **Put access control where the queries converge.** Everything that reaches the marts through the main
> Trino inherits Ranger's masks, row filters, and default-deny for free — the analyst, the dashboard,
> and this notebook all obey the same `mask-depression-pct-analyst` policy without any of them
> re-implementing it. That is the payoff of governing at the engine: **the policy lives once, and the
> identity of the caller — not the goodwill of each application — decides what comes back.** For the
> trusted-pipeline case where governance is *not* wanted, the `trino-noauth` bypass is the deliberate
> escape hatch, not an oversight.

Read-only throughout: this notebook only ever *read* the effect of policies that live in
`scripts/ranger_setup.py`. To change what is masked or filtered, edit that script and re-run it — never
the notebook.